In [25]:
import os
import glob 
import re 

import numpy as np 
import pandas as pd 

In [26]:
def df_to_latex(df, decimals = 1, include_header = True):

    lines = []
    print(df.index)

    if include_header:
        lines.append(' & '.join(str(c) for c in df.columns) + ' \\\\')
        
    for row in df.values:
        formatted = ' & '.join(f'{v:.{decimals}f}' for v in row)
        lines.append(formatted + ' \\\\')

    print('\n'.join(lines))


def preprocess(data_path, metrics): 
    
    df = pd.read_csv(data_path)
    df[['id', 'exp']] = df['experiment'].str.split('_', n = 1, expand = True)

    # convert to the proper units
    metrics_to_multiply = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc']
    df[metrics_to_multiply] = (df[metrics_to_multiply] * 100)
    mask = (df['dataset'] == 'cmupanoptic_3dgs') | (df['dataset'] == 'dex_ycb')
    alt_mask = (df['dataset'] == 'kubric-multiview')
    df.loc[mask, 'mte'] = df.loc[mask, 'mte'] * 100
    df.loc[alt_mask, 'mte'] = df.loc[alt_mask, 'mte'] * 13

    # pivot the df to match the format in the paper
    df_pivot = df.pivot(index='experiment', columns='dataset', values=metrics)
    df_pivot.columns = [f'{metric}_{dataset}' for metric, dataset in df_pivot.columns]

    return df_pivot

def preprocess_cotracker(data_path, metrics):
    df = pd.read_csv(data_path)
    df.columns = [c.replace('cotracker_', '') for c in df.columns]

    metrics_to_multiply = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc']
    df[metrics_to_multiply] = df[metrics_to_multiply] * 100
    mask = (df['dataset'] == 'cmupanoptic_3dgs') | (df['dataset'] == 'dex_ycb')
    alt_mask = (df['dataset'] == 'kubric-multiview')
    df.loc[mask, 'mte'] = df.loc[mask, 'mte'] * 100
    df.loc[alt_mask, 'mte'] = df.loc[alt_mask, 'mte'] * 13

    df = df.set_index('dataset')[metrics]

    # flatten to one row: metric_dataset columns
    result = {
        f'{metric}_{dataset}': df.loc[dataset, metric]
        for dataset in df.index
        for metric in metrics
    }
    return pd.DataFrame([result], index=['cotracker'])

# generate summary for the ncam ablations
def combine_ncam_summaries(prefix, summary_name='summary_metrics.csv', 
                           out_name='summary.csv'):

    dfs = []

    for path in sorted(glob.glob(os.path.join(prefix, 'nviews-*', '*', summary_name))):
        n_views = int(re.search(r'nviews-(\d+)', path).group(1))
        df = pd.read_csv(path)
        df.insert(0, 'n_views', n_views)
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True).sort_values('n_views').reset_index(drop=True)
    out_path = os.path.join(prefix, out_name)
    combined.to_csv(out_path, index=False)
    print(f'Saved {len(combined)} rows to {out_path}')

    return combined

import json

def load_results_df(root, agg='mean'):
    """Scan model run dirs under root and return a wide-format summary DataFrame.

    Each subdir must contain a summary.json written by eval_testset_metrics.py.
    Returns one row per experiment, with columns named {metric}_{dataset}.
    Datasets with 0 trials are included with NaN values.

    Args:
        root: path like '/groups/karashchuk/home/ruppk2/results/paper/v1'
        agg:  'mean' or 'median'
    """
    suffix = f'_{agg}'
    rows = {}
    for exp_dir in sorted(os.listdir(root)):
        summary_path = os.path.join(root, exp_dir, 'summary.json')
        if not os.path.exists(summary_path):
            continue
        with open(summary_path) as f:
            summary = json.load(f)
        row = {}
        for dataset, stats in summary.items():
            if not dataset:
                continue
            for key, val in stats.items():
                if key.endswith(suffix):
                    metric = key[: -len(suffix)]
                    row[f'{metric}_{dataset}'] = val
        rows[exp_dir] = row
    df = pd.DataFrame.from_dict(rows, orient='index')
    df.index.name = 'experiment'
    
    return df

def preprocess_posetail(prefix, metrics, datasets):

    df = load_results_df(prefix, agg = 'mean')
    df.to_csv(os.path.join(prefix, 'summary.csv'))
    print(datasets)
    
    for metric in metrics:
        for dataset in datasets:
            col = f'{metric}_{dataset}'
            if col not in df.columns:
                continue

            if metric == 'mte':
                scale = 13 if 'kubric' in dataset else 100 if dataset in ('dex_ycb', 'cmupanoptic_3dgs') else 1
            else:                       # avg_jaccard, delta_x_avg, occlusion_acc -> percent
                scale = 100

            df[col] *= scale

    return df 


def preprocess_mvtracker(run_path, metrics, datasets):

    dataset_map = {
        'dex-ycb-multiview': 'dex_ycb',
        'panoptic-multiview': 'cmupanoptic_3dgs',
        'kubric-multiview': 'kubric-multiview',
    }
    
    with open(os.path.join(run_path, 'summary.json')) as f:
        summary = json.load(f)

    row = {}
    for raw_dataset, stats in summary.items():
        dataset = dataset_map.get(raw_dataset, raw_dataset)
        if dataset not in datasets:
            continue
        for metric in metrics:
            key = f'{metric}_mean'
            if key in stats:
                row[f'{metric}_{dataset}'] = stats[key]

    df = pd.DataFrame([row])

    for metric in metrics:
        for dataset in datasets:
            col = f'{metric}_{dataset}'
            if col not in df.columns:
                continue

            if metric == 'mte':
                scale = 13 if 'kubric' in dataset else 100 if dataset in ('dex_ycb', 'cmupanoptic_3dgs') else 1
            else:                       # avg_jaccard, delta_x_avg, occlusion_acc -> percent
                scale = 100
                
            df[col] *= scale

    return df


### data loading and preprocessing

In [28]:
# LOAD 4 CAM DATA 
# load cotracker data 
cotracker_summary_path = '/groups/karashchuk/home/ruppk2/results/paper-final-v2/cotracker-outputs/paper-canonical-cached/combined_summary_metrics.csv'
cotracker_summary_path_po = '/groups/karashchuk/home/ruppk2/results/paper/cotracker-outputs/paper-4cams/combined_summary_metrics.csv'
cotracker_df = pd.read_csv(cotracker_summary_path)
cotracker_df_po = pd.read_csv(cotracker_summary_path_po)
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
cotracker_df = preprocess_cotracker(cotracker_summary_path, metrics)
cotracker_df_po = preprocess_cotracker(cotracker_summary_path_po, metrics)

# load mvtracker data (4 cams duster + gt)
datasets = ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview']
datasets_po = ['po-animal-sub256-motion', 'point-odyssey-human']

# mvtracker_path_duster = '/groups/karashchuk/home/ruppk2/results/paper-final-v2/mvtracker-outputs/native-canonical-duster-nativeres-vis'
# mvtracker_path_duster_ani = '/groups/karashchuk/home/ruppk2/results/paper/mvtracker-outputs/po-animal-sub256-motion-4cams-duster'
# mvtracker_path_duster_hum = '/groups/karashchuk/home/ruppk2/results/paper/mvtracker-outputs/point-odyssey-human-4cams-duster'
# mvtracker_df_duster = preprocess_mvtracker(mvtracker_path_duster, metrics, datasets)
# mvtracker_df_duster_ani = preprocess_mvtracker(mvtracker_path_duster_ani, metrics, datasets_po)
# mvtracker_df_duster_hum = preprocess_mvtracker(mvtracker_path_duster_hum, metrics, datasets_po)
# mvtracker_df_duster = pd.concat([mvtracker_df_duster, mvtracker_df_duster_ani, mvtracker_df_duster_hum], axis=1)

# mvtracker_path_gt = '/groups/karashchuk/home/ruppk2/results/paper-final-v2/mvtracker-outputs/native-canonical-gt-nativeres-vis'
# mvtracker_path_gt_ani = '/groups/karashchuk/home/ruppk2/results/paper/mvtracker-outputs/po-animal-sub256-motion-4cams-gt'
# mvtracker_path_gt_hum = '/groups/karashchuk/home/ruppk2/results/paper/mvtracker-outputs/point-odyssey-human-4cams-gt'
# mvtracker_df_gt = preprocess_mvtracker(mvtracker_path_gt, metrics, datasets)
# mvtracker_df_gt_ani = preprocess_mvtracker(mvtracker_path_gt_ani, metrics, datasets_po)
# mvtracker_df_gt_hum = preprocess_mvtracker(mvtracker_path_gt_hum, metrics, datasets_po)
# mvtracker_df_gt = pd.concat([mvtracker_df_gt, mvtracker_df_gt_ani, mvtracker_df_gt_hum], axis=1)

# load posetail data
# datasets = ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview', 
#             'kubric-multiview-moving', 'po-animal-moving-sub256-motion',
#             'po-animal-sub256-motion', 'po-human-moving-small', 
#             'point-odyssey-human', 'tapvid2d', 'tapvid3d_adt', 
#             'tapvid3d_drivetrack', 'tapvid3d_pstudio']
datasets = ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview', 
            'po-animal-sub256-motion', 'point-odyssey-human']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
main_prefix = '/groups/karashchuk/home/ruppk2/results/paper/posetail-outputs/'
main_prefix_final = '/groups/karashchuk/home/ruppk2/results/paper-final-v2/posetail-outputs/'
df = preprocess_posetail(main_prefix, metrics, datasets)
df_final = preprocess_posetail(main_prefix_final, metrics, datasets)
df_final.index = df_final.index + '_final'
df = pd.concat([df, df_final])

['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview', 'po-animal-sub256-motion', 'point-odyssey-human']
['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview', 'po-animal-sub256-motion', 'point-odyssey-human']


In [22]:
# LOAD ALL CAM DATA 
# load cotracker data
datasets_po = ['po-animal-sub256-motion', 'point-odyssey-human']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
cols = [f'{metric}_{dataset}' for dataset in datasets_po for metric in metrics]
 
 # load cotracker_data
cotracker_summary_path = '/groups/karashchuk/home/ruppk2/results/paper/cotracker-outputs/paper-allcams/combined_summary_metrics.csv'
cotracker_df_all = pd.read_csv(cotracker_summary_path)
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
cotracker_df_all = preprocess_cotracker(cotracker_summary_path, metrics)

# load mvtracker data (all cams duster + gt)
mvtracker_path_duster_ani = '/groups/karashchuk/home/ruppk2/results/paper/mvtracker-outputs/po-animal-sub256-motion-allcams-duster'
mvtracker_path_duster_hum = '/groups/karashchuk/home/ruppk2/results/paper/mvtracker-outputs/point-odyssey-human-allcams-duster'
mvtracker_df_duster_ani = preprocess_mvtracker(mvtracker_path_duster_ani, metrics, datasets_po)
mvtracker_df_duster_hum = preprocess_mvtracker(mvtracker_path_duster_hum, metrics, datasets_po)
mvtracker_df_duster_all = pd.concat([mvtracker_df_duster_ani, mvtracker_df_duster_hum], axis=1)

mvtracker_path_gt_ani = '/groups/karashchuk/home/ruppk2/results/paper/mvtracker-outputs/po-animal-sub256-motion-allcams-gt'
mvtracker_path_gt_hum = '/groups/karashchuk/home/ruppk2/results/paper/mvtracker-outputs/point-odyssey-human-allcams-gt'
mvtracker_df_gt_ani = preprocess_mvtracker(mvtracker_path_gt_ani, metrics, datasets_po)
mvtracker_df_gt_hum = preprocess_mvtracker(mvtracker_path_gt_hum, metrics, datasets_po)
mvtracker_df_gt_all = pd.concat([mvtracker_df_gt_ani, mvtracker_df_gt_hum], axis=1)

# load posetail data
main_prefix = '/groups/karashchuk/home/ruppk2/results/paper/posetail-outputs/'
df_all = preprocess_posetail(main_prefix, metrics, datasets)
df_all = df_all.loc[['main_finetune_vzkjqrys_allcams'], cols]


['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview', 'po-animal-sub256-motion', 'point-odyssey-human']


In [23]:
# ncam ablation data 

# cotracker 
cotracker_ncam_path = '/groups/karashchuk/home/ruppk2/results/paper/cotracker-outputs/ncam-ablation-dexycb/ncam_ablation_summary.csv'
df_ncam_cotracker = pd.read_csv(cotracker_ncam_path)
avg_jaccard_cotracker = np.round(df_ncam_cotracker['cotracker_avg_jaccard'] * 100, 1)
avg_jaccard_cotracker.index = df_ncam_cotracker['n_views']
print(list(avg_jaccard_cotracker))

# mvtracker 
mvtracker_ncam_path = '/groups/karashchuk/home/ruppk2/results/paper/mvtracker-outputs/ncam-ablation-dexycb/ncam_ablation_summary.csv'
df_ncam_mvtracker = pd.read_csv(mvtracker_ncam_path)
avg_jaccard_mvtracker = np.round(df_ncam_mvtracker['mvtracker_avg_jaccard'] * 100, 1)
avg_jaccard_mvtracker.index = df_ncam_mvtracker['n_views']
print(list(avg_jaccard_mvtracker))

# posetail
posetail_ncam_path = '/groups/karashchuk/home/ruppk2/results/paper/posetail-outputs/ncam_ablation_dexycb/ncam_ablation_summary.csv'
df_ncam_posetail = pd.read_csv(posetail_ncam_path)
avg_jaccard_posetail = np.round(df_ncam_posetail['avg_jaccard'] * 100, 1)
avg_jaccard_posetail.index = df_ncam_posetail['n_views']
print(list(avg_jaccard_posetail))

# save to df for figure
df_ncam = pd.concat((avg_jaccard_cotracker, avg_jaccard_mvtracker, avg_jaccard_posetail), axis = 1, sort = True)
df_ncam.columns = ['CoTracker3 Baseline', 'MVTracker Baseline', 'Tracktail']
df_ncam.index.name = 'n_views'
df_ncam.to_csv('dexycb_ncam_results.csv')

[46.4, 53.3, 61.6, 63.2, 68.9, 70.3, 70.9]
[55.9, 48.0, 61.5, 69.4, 65.2, 69.0, 69.4, 70.6]
[44.5, 74.1, 77.7, 75.6, 77.5, 84.6, 86.7, 87.6]


### main table (table 1)

In [32]:
# main table
experiments = ['main_finetune_vzkjqrys_canonical_cached_final']
datasets = ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]

# cotracker baseline main metrics (4 cams)
print('cotracker')
sub_df_cotracker = cotracker_df[cols]
df_to_latex(sub_df_cotracker, decimals = 1, include_header = True)
print('')

# mvtracker (4 cams duster)
print('mvtracker, duster')
sub_df_mvtracker_duster = mvtracker_df_duster[cols]
df_to_latex(sub_df_mvtracker_duster, decimals = 1, include_header = True)
print('')

# mvtracker (4 cams gt)
print('mvtracker, gt')
datasets_mvtracker_gt = ['cmupanoptic_3dgs', 'kubric-multiview']
cols_mvtracker_gt = [f'{metric}_{dataset}' for dataset in datasets_mvtracker_gt for metric in metrics]
sub_df_mvtracker_gt = mvtracker_df_gt[cols_mvtracker_gt]
df_to_latex(sub_df_mvtracker_gt, decimals = 1, include_header = True)
print('')

# posetail (4 cams)
print('tracktail')
sub_df_posetail = df.loc[experiments, cols]
df_to_latex(sub_df_posetail, decimals = 1, include_header = True)

cotracker
Index(['cotracker'], dtype='object')
avg_jaccard_cmupanoptic_3dgs & delta_x_avg_cmupanoptic_3dgs & occlusion_acc_cmupanoptic_3dgs & mte_cmupanoptic_3dgs & avg_jaccard_dex_ycb & delta_x_avg_dex_ycb & occlusion_acc_dex_ycb & mte_dex_ycb & avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
62.5 & 70.3 & 81.5 & 23.2 & 66.4 & 73.2 & 84.4 & 6.0 & 42.9 & 48.9 & 74.5 & 17.9 \\

tracktail
Index(['main_finetune_vzkjqrys_canonical_cached_final'], dtype='object', name='experiment')
avg_jaccard_cmupanoptic_3dgs & delta_x_avg_cmupanoptic_3dgs & occlusion_acc_cmupanoptic_3dgs & mte_cmupanoptic_3dgs & avg_jaccard_dex_ycb & delta_x_avg_dex_ycb & occlusion_acc_dex_ycb & mte_dex_ycb & avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
80.7 & 95.4 & 88.1 & 2.7 & 83.3 & 93.9 & 91.0 & 0.6 & 76.9 & 86.9 & 91.5 & 1.1 \\


### point odyssey

In [66]:
# PO benchmark 
experiments = ['main_finetune_vzkjqrys']
# datasets = ['point-odyssey-human', 'po-animal-sub256-motion',
#             'po-animal-moving-sub256-motion','po-human-moving-small']
datasets = ['point-odyssey-human', 'po-animal-sub256-motion']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]


# COTRACKER (4 cams, 8 cams)
print('cotracker')
df_to_latex(cotracker_df_po[cols], decimals = 1, include_header = True)
df_to_latex(cotracker_df_all[cols], decimals = 1, include_header = True)
print('')

# MVTRACKER duster depth (4 cams, 8 cams)
print('mvtracker, duster')
df_to_latex(mvtracker_df_duster[cols], decimals = 1, include_header = True)
df_to_latex(mvtracker_df_duster_all[cols], decimals = 1, include_header = True)
print('')

# MVTRACKER gt depth (4 cams, 8 cams)
print('mvtracker, gt')
df_to_latex(mvtracker_df_gt[cols], decimals = 1, include_header = True)
df_to_latex(mvtracker_df_gt_all[cols], decimals = 1, include_header = True)

print('tracktail')
df_4 = df.loc[experiments, cols]
df_to_latex(df_4, decimals = 1, include_header = True)
df_to_latex(df_all, decimals = 1, include_header = True)
print('') 


cotracker
Index(['cotracker'], dtype='object')
avg_jaccard_point-odyssey-human & delta_x_avg_point-odyssey-human & occlusion_acc_point-odyssey-human & mte_point-odyssey-human & avg_jaccard_po-animal-sub256-motion & delta_x_avg_po-animal-sub256-motion & occlusion_acc_po-animal-sub256-motion & mte_po-animal-sub256-motion \\
21.6 & 23.0 & 50.1 & 15.8 & 24.0 & 30.3 & 50.4 & 10.3 \\
Index(['cotracker'], dtype='object')
avg_jaccard_point-odyssey-human & delta_x_avg_point-odyssey-human & occlusion_acc_point-odyssey-human & mte_point-odyssey-human & avg_jaccard_po-animal-sub256-motion & delta_x_avg_po-animal-sub256-motion & occlusion_acc_po-animal-sub256-motion & mte_po-animal-sub256-motion \\
23.8 & 26.7 & 68.4 & 17.8 & 27.8 & 34.1 & 70.8 & 7.2 \\

mvtracker, duster
RangeIndex(start=0, stop=1, step=1)
avg_jaccard_point-odyssey-human & delta_x_avg_point-odyssey-human & occlusion_acc_point-odyssey-human & mte_point-odyssey-human & avg_jaccard_po-animal-sub256-motion & delta_x_avg_po-animal-sub2

### ablation studies

In [41]:
# cam self attention ablation
# experiments = ['c3bv8sk5_ablation_cam_self_attn', '2p9of8sk_main_pretrain']
experiments = ['ablation_cam_self_attn_folhop3e', 'main_pretrain_e25w9v86']
datasets = ['kubric-multiview']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']

cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
sub_df = df.loc[experiments, cols]
df_to_latex(sub_df, decimals = 1, include_header = True)
sub_df

Index(['ablation_cam_self_attn_folhop3e', 'main_pretrain_e25w9v86'], dtype='object', name='experiment')
avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
36.7 & 49.6 & 83.9 & 6.0 \\
48.3 & 62.0 & 86.8 & 3.4 \\


,avg_jaccard_kubric-multiview,delta_x_avg_kubric-multiview,occlusion_acc_kubric-multiview,mte_kubric-multiview
experiment,,,,
ablation_cam_self_attn_folhop3e,36.705671,49.630045,83.910428,6.014933
main_pretrain_e25w9v86,48.280559,62.031236,86.756263,3.373463


In [70]:
# dataset ablation 
# experiments = ['l60u31gr_ablation_dataset_kubric_only', 'r2eufrxl_ablation_dataset_static_only', '2p9of8sk_main_pretrain']
experiments = ['ablation_dataset_static_kub_only_f4wpvece', 'main_pretrain_e25w9v86']
datasets = ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']

cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
sub_df = df.loc[experiments, cols]
df_to_latex(sub_df, decimals = 1, include_header = True)
sub_df

Index(['ablation_dataset_static_kub_only_f4wpvece', 'main_pretrain_e25w9v86'], dtype='object', name='experiment')
avg_jaccard_cmupanoptic_3dgs & delta_x_avg_cmupanoptic_3dgs & occlusion_acc_cmupanoptic_3dgs & mte_cmupanoptic_3dgs & avg_jaccard_dex_ycb & delta_x_avg_dex_ycb & occlusion_acc_dex_ycb & mte_dex_ycb & avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
34.6 & 74.6 & 53.6 & 8.9 & 70.9 & 89.8 & 85.8 & 1.2 & 53.3 & 67.7 & 87.9 & 2.5 \\
33.9 & 79.8 & 50.0 & 11.7 & 71.5 & 90.7 & 86.5 & 1.0 & 48.3 & 62.0 & 86.8 & 3.4 \\


,avg_jaccard_cmupanoptic_3dgs,delta_x_avg_cmupanoptic_3dgs,occlusion_acc_cmupanoptic_3dgs,mte_cmupanoptic_3dgs,avg_jaccard_dex_ycb,delta_x_avg_dex_ycb,occlusion_acc_dex_ycb,mte_dex_ycb,avg_jaccard_kubric-multiview,delta_x_avg_kubric-multiview,occlusion_acc_kubric-multiview,mte_kubric-multiview
experiment,,,,,,,,,,,,
ablation_dataset_static_kub_only_f4wpvece,34.550994,74.608263,53.585191,8.942008,70.887391,89.78439,85.789585,1.220539,53.267468,67.719843,87.899797,2.481085
main_pretrain_e25w9v86,33.856086,79.820163,50.002108,11.697668,71.507112,90.65675,86.477918,0.988727,48.280559,62.031236,86.756263,3.373463


In [47]:
# encoder size ablation
# experiments = ['aib22abs_ablation_encoder_size_base', '2p9of8sk_main_pretrain',
#               'wu7v94nd_ablation_encoder_size_giant', 'ab3ei96j_ablation_encoder_size_gigantic']
experiments = ['ablation_encoder_size_base_hv8nguwx', 'main_pretrain_e25w9v86', 'ablation_encoder_size_giant_ubk4lvez', 'ablation_encoder_size_gigantic_u1c34ou7']
datasets = ['kubric-multiview']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']

cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
sub_df = df.loc[experiments, cols]
df_to_latex(sub_df, decimals = 1, include_header = True)
sub_df

Index(['ablation_encoder_size_base_hv8nguwx', 'main_pretrain_e25w9v86',
       'ablation_encoder_size_giant_ubk4lvez',
       'ablation_encoder_size_gigantic_u1c34ou7'],
      dtype='object', name='experiment')
avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
44.3 & 57.7 & 85.8 & 4.2 \\
48.3 & 62.0 & 86.8 & 3.4 \\
49.3 & 63.0 & 87.0 & 3.2 \\
51.9 & 65.8 & 87.6 & 2.9 \\


,avg_jaccard_kubric-multiview,delta_x_avg_kubric-multiview,occlusion_acc_kubric-multiview,mte_kubric-multiview
experiment,,,,
ablation_encoder_size_base_hv8nguwx,44.311886,57.734312,85.783624,4.161889
main_pretrain_e25w9v86,48.280559,62.031236,86.756263,3.373463
ablation_encoder_size_giant_ubk4lvez,49.273280,63.008889,87.036557,3.242361
ablation_encoder_size_gigantic_u1c34ou7,51.878901,65.832156,87.630080,2.903080


In [66]:
# latent dim ablation
# experiments = ['hxgceji7_ablation_latent_dim_256', '2p9of8sk_main_pretrain', 'c8a16qaw_ablation_latent_dim_1024']
experiments = ['ablation_latent_dim_256_lczwsdx3', 'main_pretrain_e25w9v86', 'ablation_latent_dim_1024_wk2i4o1u']
datasets = ['kubric-multiview']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']

cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
sub_df = df.loc[experiments, cols]
df_to_latex(sub_df, decimals = 1, include_header = True)
sub_df

Index(['ablation_latent_dim_256_lczwsdx3', 'main_pretrain_e25w9v86',
       'ablation_latent_dim_1024_wk2i4o1u'],
      dtype='object', name='experiment')
avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
46.0 & 59.5 & 86.4 & 3.7 \\
48.3 & 62.0 & 86.8 & 3.4 \\
48.5 & 62.1 & 86.9 & 3.4 \\


,avg_jaccard_kubric-multiview,delta_x_avg_kubric-multiview,occlusion_acc_kubric-multiview,mte_kubric-multiview
experiment,,,,
ablation_latent_dim_256_lczwsdx3,45.965104,59.534358,86.364883,3.693987
main_pretrain_e25w9v86,48.280559,62.031236,86.756263,3.373463
ablation_latent_dim_1024_wk2i4o1u,48.463731,62.098036,86.894706,3.418680


### tapvid3d

In [ ]:
# # comparison to d4rt on tapvid3d
# experiments = ['main_finetune_vzkjqrys_final']
# datasets = ['tapvid3d_drivetrack', 'tapvid3d_adt', 'tapvid3d_pstudio']
# metrics = ['avg_jaccard_cam', 'occlusion_acc_cam'] #TODO: check ap3d/L1 metric they use

# cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
# sub_df = df.loc[experiments, cols]

# # print('cotracker')
# # df_to_latex(cotracker_df[cols], decimals = 1, include_header = True)

# print('tracktail')
# df_to_latex(sub_df, decimals = 1, include_header = True)
# sub_df

KeyError: "['occlusion_acc_cam_tapvid3d_drivetrack', 'occlusion_acc_cam_tapvid3d_adt', 'occlusion_acc_cam_tapvid3d_pstudio'] not in index"

In [54]:
exp = ['main_finetune_vzkjqrys_final']
ds_cam   = ['tapvid3d_drivetrack', 'tapvid3d_adt', 'tapvid3d_pstudio']   # D4RT column order
ds_world = ['tapvid3d_drivetrack', 'tapvid3d_adt']

cam_cols   = [f'{m}_{d}' for d in ds_cam   for m in ['avg_jaccard_cam', 'delta_x_avg_cam', 'occlusion_acc']]
world_cols = [f'{m}_{d}' for d in ds_world for m in ['delta_x_avg_world', 'l1_world']]

df_to_latex(df.loc[exp, cam_cols],   decimals = 3, include_header = True)
df_to_latex(df.loc[exp, world_cols], decimals = 3, include_header = True)


Index(['main_finetune_vzkjqrys_final'], dtype='object', name='experiment')
avg_jaccard_cam_tapvid3d_drivetrack & delta_x_avg_cam_tapvid3d_drivetrack & occlusion_acc_tapvid3d_drivetrack & avg_jaccard_cam_tapvid3d_adt & delta_x_avg_cam_tapvid3d_adt & occlusion_acc_tapvid3d_adt & avg_jaccard_cam_tapvid3d_pstudio & delta_x_avg_cam_tapvid3d_pstudio & occlusion_acc_tapvid3d_pstudio \\
0.246 & 0.499 & 0.423 & 0.129 & 0.456 & 0.202 & 0.427 & 0.725 & 0.628 \\
Index(['main_finetune_vzkjqrys_final'], dtype='object', name='experiment')
delta_x_avg_world_tapvid3d_drivetrack & l1_world_tapvid3d_drivetrack & delta_x_avg_world_tapvid3d_adt & l1_world_tapvid3d_adt \\
0.510 & 0.343 & 0.459 & 1.374 \\


In [ ]:
# # comparison to mvtracker on tapvid2d
# # TODO: add thresholded aj and dx? to metrics
# experiments = ['TODO_main_finetune']
# datasets = ['tapvid2d']
# metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']

# cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
# sub_df = df.loc[experiments, cols]
# df_to_latex(sub_df, decimals = 1, include_header = True)
# sub_df